# Notebook 04 — Arquitectura de Predicción Multi-Distancia: Cobertura Real y Capa 2 Demográfica

**Objetivo de este notebook:**
1. Verificar qué distancias están **realmente** disponibles en cada dataset (sin asumir cobertura)
2. Formalizar la **arquitectura de 4 capas** para predicción de 5K / 10K / 21K / 42K
3. Construir y evaluar la **Capa 2 demográfica** sobre el baseline Riegel calibrado
4. Dejar claro qué queda resuelto y qué queda pendiente para los notebooks siguientes

**Datasets usados:**
- `archive (3)/Results.csv` — 429K finishers de maratón (42K), 2023
- `Nuevo dataset Project_2-Marathon-Predictor/marathon_results_2015-2018.csv` — Boston Marathon con splits por cada 5K
- `16620238/*.parquet` — historial de entrenamiento semanal (sin tiempos de carrera)

**Posición en la arquitectura:**
```
Notebook 01 → EDA + Riegel baseline (completado)
Notebook 02 → CTL/ATL/ACWR feature engineering (completado)
Notebook 03 → Calibración empírica Riegel con Boston (completado)
Notebook 04 → [ESTE] Arquitectura formal + Capa 2 demográfica
Notebook 05 → Capa 3: corrección por carga (pending)
```

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# ─── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ROOT         = NOTEBOOK_DIR.parent.parent   # running_coaching/
DATA_DIR     = ROOT / 'Datasets running'
FIGS_DIR     = ROOT / 'ml' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))

# ─── Libraries ────────────────────────────────────────────────────────────────
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.linear_model   import Ridge
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics         import mean_absolute_error

# ─── Project modules ──────────────────────────────────────────────────────────
from src.ml.riegel import (
    riegel, riegel_calibrated,
    CALIBRATED_EXPONENTS, CALIBRATED_MAE_MIN,
)

print('ROOT      :', ROOT)
print('DATA_DIR  :', DATA_DIR)
print('FIGS_DIR  :', FIGS_DIR)
print('DATA exists:', DATA_DIR.exists())

---
## 1. Verificación de cobertura real de distancias

> **Regla metodológica:** nunca asumir cobertura. Verificar con los datos antes de diseñar el modelo.

In [ ]:
# ─── 1a. Results.csv ─────────────────────────────────────────────────────────
RESULTS_PATH = DATA_DIR / 'archive (3)' / 'Results.csv'
df_results   = pd.read_csv(RESULTS_PATH)

print('=== Results.csv ===')
print(f'Shape     : {df_results.shape}')
print(f'Columns   : {df_results.columns.tolist()}')
print(f'\nTop 10 races por frecuencia:')
print(df_results['Race'].value_counts().head(10).to_string())

# Buscar keywords de distancias cortas en el nombre de la carrera
races = df_results['Race'].unique()
print('\n--- Búsqueda de keywords de distancia ---')
for kw in ['5K', '5k', '10K', '10k', 'Half', 'half', 'HM', '21']:
    hits = [r for r in races if kw in str(r)]
    print(f"  '{kw}': {len(hits)} carreras únicas — ejemplos: {hits[:2]}")

print(f'\nFinish (seg): min={df_results["Finish"].min():.0f}  '
      f'median={df_results["Finish"].median():.0f}  '
      f'max={df_results["Finish"].max():.0f}')
print(f'  → {df_results["Finish"].min()/3600:.2f}h  '
      f'median={df_results["Finish"].median()/3600:.2f}h  '
      f'max={df_results["Finish"].max()/3600:.2f}h')
print(f'\nGénero: {df_results["Gender"].value_counts().to_dict()}')
print(f'Edad  : min={df_results["Age"].min():.0f}  '
      f'mean={df_results["Age"].mean():.1f}  '
      f'max={df_results["Age"].max():.0f}')

In [ ]:
# ─── 1b. Boston Marathon 2015-2018: splits por cada 5K ───────────────────────
BOSTON_DIR  = DATA_DIR / 'Nuevo dataset Project_2-Marathon-Predictor'
boston_dfs  = []

for yr in [2015, 2016, 2017, 2018]:
    fpath = BOSTON_DIR / f'marathon_results_{yr}.csv'
    df_yr = pd.read_csv(fpath, low_memory=False)
    df_yr['year'] = yr
    boston_dfs.append(df_yr)
    print(f'{yr}: {df_yr.shape}  columns={df_yr.columns.tolist()}')

df_boston = pd.concat(boston_dfs, ignore_index=True)
print(f'\nDataset combinado: {df_boston.shape}')
print(f'Columnas de splits disponibles: {[c for c in df_boston.columns if c in ["5K","10K","15K","20K","Half","25K","30K","35K","40K","Official Time","Pace","Proj Time"]]}')
print(f'\nSample de tiempos (3 corredores):')
print(df_boston[['Name','Age','M/F','5K','10K','Half','Official Time','year']].dropna().head(3).to_string())

In [ ]:
# ─── 1c. 16620238: verificar que NO hay tiempos de carrera ───────────────────
parquet_dir = DATA_DIR / '16620238'
parquet_files = sorted(parquet_dir.glob('*.parquet'))
print(f'Archivos parquet: {[f.name for f in parquet_files]}')

# Cargar el semanal 2020 (más compacto que el diario)
df_16 = pd.read_parquet(parquet_dir / 'run_ww_2020_w.parquet')
print(f'\nrun_ww_2020_w.parquet: {df_16.shape}')
print(f'Columnas y tipos:')
print(df_16.dtypes.to_string())
print(f'\nMuestra:')
print(df_16.head(3).to_string())

# Buscar columnas candidatas a "tiempo de carrera"
race_candidates = [c for c in df_16.columns if any(
    kw in c.lower() for kw in ['race','finish','pr','pb','best','result','5k','10k','21','42']
)]
print(f'\nCandidatos a "tiempo de carrera": {race_candidates}')

# Columna 'major': ¿qué eventos aparecen?
if 'major' in df_16.columns:
    print(f'\nValores únicos en "major" (primeros 20):')
    print(df_16['major'].value_counts().head(20).to_string())

### Síntesis de cobertura real

| Dataset | Distancias disponibles | Tipo de dato | Uso en el modelo |
|---|---|---|---|
| **Results.csv** (429K) | **42K únicamente** | Tiempo final | Prior demográfico: E[T_42K \| age, gender] |
| **Boston 2015-2018** (103K) | 5K, 10K, 15K, 20K, **21K**, 25K, 30K, 35K, 40K, **42K** | Splits acumulados + tiempo oficial | Corrección residuos Riegel por edad × género |
| **16620238** (1.9M/semana) | Distancia de entrenamiento semanal (≠ tiempo de carrera) | Carga semanal de entrenamiento | Features CTL/ATL/ACWR para Capa 3 |
| **pmdata/p01** | 1 atleta, HR + sesiones | Individual | Demostración cualitativa (no ML) |

**Hallazgo crítico:** Los splits del Boston Marathon (5K, 10K, Half en ruta del maratón) son la única fuente que tiene tiempos intermedios + tiempo final **del mismo corredor** en múltiples distancias.

> ⚠️ **Caveat metodológico importante:** los splits a 5K y 10K dentro de un maratón NO son equivalentes a un PR de 5K o 10K en carrera independiente. En los primeros km de un maratón, los corredores van deliberadamente más lentos (pacing strategy). El split a 5K de alguien que termina en 4h es ~30-35 min, pero su PR de 5K real podría ser 24-27 min. **Esto debe quedar explícito en la tesis.**

---
## 2. Arquitectura formal de predicción — 4 capas

Con base en la cobertura real de datos, la arquitectura queda fijada así:

In [ ]:
architecture = '''
╔══════════════════════════════════════════════════════════════════════════╗
║         ARQUITECTURA DE PREDICCIÓN MULTI-DISTANCIA (5K/10K/21K/42K)    ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  INPUT: perfil del atleta + PRs disponibles + carga + check-in          ║
║                                                                          ║
║  ┌──────────────────────────────────────────────────────────────┐       ║
║  │  CAPA 1 — Baseline Riegel calibrado por segmento             │       ║
║  │  T2 = T1 × (D2/D1)^exp_seg  donde exp_seg ∈ {1.0332–1.11}   │       ║
║  │  MAE: 2.5–12 min (según segmento)  [COMPLETADO — NB03]       │       ║
║  │  Activa cuando: hay al menos 1 PR disponible                  │       ║
║  └──────────────────┬───────────────────────────────────────────┘       ║
║                     │ + residual demográfico                            ║
║  ┌──────────────────▼───────────────────────────────────────────┐       ║
║  │  CAPA 2 — Corrección demográfica                             │       ║
║  │  2a: Prior poblacional → E[T_42K | age_group, gender]        │       ║
║  │      Fuente: Results.csv (429K corredores, 2023)             │       ║
║  │      Usa cuando: NO hay PR (fallback)                        │       ║
║  │  2b: Corrección de residuos Riegel → ΔT(age, gender)         │       ║
║  │      Fuente: Boston 2015-2018 (103K, splits + oficial)       │       ║
║  │      Usa cuando: SÍ hay PR (ajuste fino sobre Capa 1)        │       ║
║  │  [ESTE NOTEBOOK]                                              │       ║
║  └──────────────────┬───────────────────────────────────────────┘       ║
║                     │ + corrección por carga de entrenamiento           ║
║  ┌──────────────────▼───────────────────────────────────────────┐       ║
║  │  CAPA 3 — Corrección por carga (CTL / ATL / ACWR)           │       ║
║  │  ΔT_load = f(CTL, ATL, TSB, ACWR, km_recientes)             │       ║
║  │  Pendiente: necesita dataset con carga + tiempo de carrera   │       ║
║  │  Estrategia: Run Club (40K, evaluar sintéticos) o atleta     │       ║
║  │  propio con Strava + PRs reales                              │       ║
║  └──────────────────┬───────────────────────────────────────────┘       ║
║                     │ + ajuste de bienestar                            ║
║  ┌──────────────────▼───────────────────────────────────────────┐       ║
║  │  CAPA 4 — Ajuste por bienestar (check-in de la semana)       │       ║
║  │  ΔT_wellness = g(fatiga, dolor, sueño, estrés)               │       ║
║  │  Pendiente: requiere datos longitudinales del atleta en app  │       ║
║  └──────────────────┬───────────────────────────────────────────┘       ║
║                     │                                                   ║
║  OUTPUT: estimate_sec ± MAE_esperado  por distancia objetivo           ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
'''

print(architecture)

# Estado actual por distancia × capa
estado = pd.DataFrame({
    'Distancia': ['42K', '21K', '10K', '5K'],
    'Capa 1 (Riegel)':       ['✅ Calibrado',    '✅ Calibrado',    '✅ Calibrado',    '✅ Calibrado'],
    'Capa 2a (Prior demog)': ['✅ Results.csv',   '⚠️ Sin datos',    '⚠️ Sin datos',    '⚠️ Sin datos'],
    'Capa 2b (Correc.demog)':['✅ Boston splits', '✅ Boston splits', '⚠️ Proxy*',       '⚠️ Proxy*'],
    'Capa 3 (Carga)':        ['⚠️ Pendiente',    '⚠️ Pendiente',    '⚠️ Pendiente',    '⚠️ Pendiente'],
    'Capa 4 (Wellness)':     ['⚠️ Pendiente',    '⚠️ Pendiente',    '⚠️ Pendiente',    '⚠️ Pendiente'],
})
print('Estado actual por distancia × capa:')
print(estado.to_string(index=False))
print('\n* Proxy = splits de Boston dentro del maratón ≠ PR independiente')

---
## 3. Capa 2a — Prior demográfico poblacional (Results.csv)

**Propósito:** cuando el atleta NO tiene ningún PR disponible, el sistema puede dar una estimación usando únicamente su edad y género, comparado con la distribución poblacional real de 429K corredores.

**Limitación:** alta varianza individual dentro de cada grupo → MAE ~35–50 min. Útil como **fallback**, no como predicción precisa.

In [ ]:
# ─── Filtrar datos válidos ────────────────────────────────────────────────────
df_valid = df_results[
    (df_results['Gender'].isin(['M', 'F'])) &
    (df_results['Age'].between(18, 75)) &
    (df_results['Finish'].between(7200, 21600))   # 2h – 6h
].copy()

pct_kept = len(df_valid) / len(df_results)
print(f'Filas originales : {len(df_results):,}')
print(f'Filas filtradas  : {len(df_valid):,}  ({pct_kept:.1%} del total)')
print(f'Excluidos        : edad fuera [18,75], género U/X, tiempos outlier (<2h o >6h)')

# ─── Grupos de edad ──────────────────────────────────────────────────────────
AGE_BINS   = [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, 76]
AGE_LABELS = ['18-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59','60-64','65+']

df_valid['age_group'] = pd.cut(
    df_valid['Age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
)

# ─── Tabla de referencia demográfica ─────────────────────────────────────────
demo_table = (
    df_valid
    .groupby(['age_group', 'Gender'], observed=True)['Finish']
    .agg(mediana='median', media='mean', p25=lambda x: x.quantile(0.25),
         p75=lambda x: x.quantile(0.75), n='count')
    .round(0)
    .unstack('Gender')
)

# Convertir segundos a hh:mm para display
def fmt_sec(s):
    if pd.isna(s): return '--'
    h, r = divmod(int(s), 3600)
    m, _ = divmod(r, 60)
    return f'{h}:{m:02d}'

print('\nMediana de tiempo de maratón (hh:mm) por grupo de edad y género:')
median_sec = demo_table['mediana']
median_fmt = median_sec.applymap(fmt_sec)
print(median_fmt.to_string())

# MAE del prior demográfico (predicción = mediana del grupo)
group_medians = df_valid.groupby(['age_group', 'Gender'], observed=True)['Finish'].median()
df_valid['demo_prediction'] = df_valid.set_index(['age_group', 'Gender']).index.map(
    lambda idx: group_medians.get(idx, np.nan)
)
df_valid['demo_prediction'] = df_valid.apply(
    lambda r: group_medians.get((r['age_group'], r['Gender']), np.nan), axis=1
)
mae_demo = mean_absolute_error(
    df_valid['Finish'].dropna(),
    df_valid['demo_prediction'].dropna(),
)
print(f'\nMAE del prior demográfico (mediana grupal): {mae_demo/60:.1f} min')
print('(Comparar con Capa 1 Riegel calibrado: ~9.2 min global)')

In [ ]:
# ─── Figura 04_01: curvas demográficas de referencia ─────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Capa 2a — Prior demográfico: distribución de tiempo de maratón\npor grupo de edad y género (Results.csv, n=429K)', fontsize=12)

colors = {'M': '#2196F3', 'F': '#E91E63'}
labels_gender = {'M': 'Hombres', 'F': 'Mujeres'}

# Ax1: mediana con IQR
for gender in ['M', 'F']:
    gdata = df_valid[df_valid['Gender'] == gender].groupby('age_group', observed=True)['Finish']
    med  = gdata.median() / 3600
    p25  = gdata.quantile(0.25) / 3600
    p75  = gdata.quantile(0.75) / 3600
    x    = range(len(med))
    ax1.plot(x, med.values, 'o-', color=colors[gender], label=f'{labels_gender[gender]} (mediana)', lw=2)
    ax1.fill_between(x, p25.values, p75.values, alpha=0.15, color=colors[gender], label=f'P25–P75')

ax1.set_xticks(range(len(AGE_LABELS)))
ax1.set_xticklabels(AGE_LABELS, rotation=30, ha='right')
ax1.set_ylabel('Tiempo (horas)')
ax1.set_xlabel('Grupo de edad')
ax1.set_title('Mediana e IQR por grupo de edad')
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{int(x)}:{int((x%1)*60):02d}'
))
ax1.grid(alpha=0.3)

# Ax2: diferencia F-M (penalty femenino)
med_m = df_valid[df_valid['Gender']=='M'].groupby('age_group', observed=True)['Finish'].median()
med_f = df_valid[df_valid['Gender']=='F'].groupby('age_group', observed=True)['Finish'].median()
diff_min = (med_f - med_m) / 60
ax2.bar(range(len(diff_min)), diff_min.values, color='#9C27B0', alpha=0.7)
ax2.set_xticks(range(len(AGE_LABELS)))
ax2.set_xticklabels(AGE_LABELS, rotation=30, ha='right')
ax2.set_ylabel('Diferencia F – M (minutos)')
ax2.set_xlabel('Grupo de edad')
ax2.set_title('Penalización femenina vs masculina (mediana)')
ax2.axhline(0, color='black', lw=0.8)
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'+{x:.0f}' if x>0 else f'{x:.0f}'))

plt.tight_layout()
fig_path = FIGS_DIR / '04_fig_01_curvas_demograficas.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path.name}')

---
## 4. Capa 2b — Corrección de residuos Riegel por demografía (Boston 2015-2018)

**Lógica:** los splits de Boston nos dan el tiempo en la media maratón (Half) y el tiempo oficial (42K) del **mismo corredor**. Aplicamos Riegel calibrado sobre el Half y medimos el error sistemático por edad y género.

Esta corrección activa cuando hay un PR disponible — ajusta el sesgo de Riegel para el grupo demográfico del atleta.

In [ ]:
# ─── Parser de tiempos tipo 'H:MM:SS' ────────────────────────────────────────
def parse_time_str(t) -> float:
    """Parse 'H:MM:SS' o '-' a segundos. Retorna np.nan si inválido."""
    if pd.isna(t) or str(t).strip() in ['-', '', 'nan', '--']:
        return np.nan
    parts = str(t).strip().split(':')
    try:
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
    except (ValueError, TypeError):
        return np.nan
    return np.nan

# ─── Parsear splits relevantes ───────────────────────────────────────────────
SPLIT_COLS = ['5K', '10K', '15K', '20K', 'Half', '25K', '30K', '35K', '40K', 'Official Time']
for col in SPLIT_COLS:
    if col in df_boston.columns:
        df_boston[f'{col}_sec'] = df_boston[col].apply(parse_time_str)

# ─── Filtrar datos válidos ────────────────────────────────────────────────────
gender_col = 'M/F' if 'M/F' in df_boston.columns else 'Gender'

df_bv = df_boston[
    df_boston['Half_sec'].notna() &
    df_boston['Official Time_sec'].notna() &
    df_boston['Age'].between(18, 75) &
    df_boston[gender_col].isin(['M', 'F'])
].copy()

# Filtros de tiempos plausibles
df_bv = df_bv[
    df_bv['Half_sec'].between(3600, 12600) &           # 1h – 3:30h en media
    df_bv['Official Time_sec'].between(7200, 21600)    # 2h – 6h en completo
].copy()

print(f'Boston válido: {len(df_bv):,} corredores  (de {len(df_boston):,} originales)')
print(f'Por año: {df_bv["year"].value_counts().sort_index().to_dict()}')

# ─── Aplicar Riegel calibrado: Half → Official ────────────────────────────────
D_HALF = 21.0975
D_FULL = 42.195

# riegel_calibrated retorna (t2_sec, segment, exponent)
riegel_results = df_bv['Half_sec'].apply(
    lambda t: riegel_calibrated(t, D_HALF, D_FULL)
)
df_bv['riegel_estimate_sec'] = riegel_results.apply(lambda r: r[0])
df_bv['riegel_segment']      = riegel_results.apply(lambda r: r[1])

# Residual: cuánto se equivoca Riegel (positivo = Riegel optimista, llegó tarde)
df_bv['riegel_residual_sec'] = df_bv['Official Time_sec'] - df_bv['riegel_estimate_sec']
df_bv['riegel_residual_min'] = df_bv['riegel_residual_sec'] / 60

mae_riegel = df_bv['riegel_residual_sec'].abs().mean() / 60
bias_riegel = df_bv['riegel_residual_sec'].mean() / 60

print(f'\nRiegel calibrado (Half→Full):')
print(f'  MAE global  : {mae_riegel:.1f} min')
print(f'  Bias global : {bias_riegel:+.1f} min  (+ = Riegel subestima, llegada más tarde que lo predicho)')
print(f'\nDistribución por segmento:')
print(df_bv.groupby('riegel_segment')['riegel_residual_min'].agg(
    n='count', mae=lambda x: x.abs().mean(), bias='mean'
).round(2).to_string())

In [ ]:
# ─── Grupos de edad para Boston ──────────────────────────────────────────────
df_bv['age_group'] = pd.cut(
    df_bv['Age'], bins=AGE_BINS, labels=AGE_LABELS, right=False
)

# Sesgo de Riegel por edad × género
print('Sesgo medio de Riegel (minutos) por edad × género:')
print('(+ = Riegel optimista: el corredor llegó más tarde de lo predicho)')
bias_table = df_bv.groupby(['age_group', gender_col], observed=True)['riegel_residual_min'].agg(
    ['mean', 'median', 'count']
).round(2).unstack(gender_col)
print(bias_table['mean'].to_string())

# ─── Modelo Ridge: predecir el residual desde age × gender ───────────────────
df_model = df_bv.dropna(subset=['Age', 'riegel_residual_sec', gender_col]).copy()
df_model['is_female'] = (df_model[gender_col] == 'F').astype(float)
df_model['age_sq']    = df_model['Age'] ** 2
df_model['age_x_f']   = df_model['Age'] * df_model['is_female']

FEATURE_NAMES = ['Age', 'age_sq', 'is_female', 'age_x_f']
X = df_model[FEATURE_NAMES].values
y = df_model['riegel_residual_sec'].values

scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X)

# GroupKFold por año (4 folds = 4 años de Boston)
groups  = df_model['year'].values
gkf     = GroupKFold(n_splits=4)
ridge   = Ridge(alpha=10.0)

cv_mae_sec = -cross_val_score(
    ridge, X_sc, y, cv=gkf, groups=groups,
    scoring='neg_mean_absolute_error'
)
cv_mae_min = cv_mae_sec / 60

print(f'\nMAE de la corrección demográfica (predicción del residual):')
print(f'  MAE medio del residual predicho : {cv_mae_min.mean():.1f} min')
print('  → el modelo no puede corregir perfectamente el sesgo (demasiada varianza individual)')

# Fit en todos los datos para analizar coeficientes
ridge.fit(X_sc, y)
feature_means = scaler.mean_
feature_std   = scaler.scale_

print(f'\nCoeficientes del modelo de corrección:')
print(f'  Intercepto (bias global)    : {ridge.intercept_/60:+.2f} min')
for name, coef, mean_, std_ in zip(FEATURE_NAMES, ridge.coef_, feature_means, feature_std):
    coef_per_unit = coef / std_ * std_   # en escala original
    print(f'  {name:20s}: {coef/60:+.3f} min (coef estandarizado)')

In [ ]:
# ─── Comparación de MAE: Riegel solo vs Riegel + corrección demográfica ───────

# Capa 1 sola: MAE de Riegel calibrado
mae_capa1 = df_bv['riegel_residual_sec'].abs().mean() / 60

# Capa 1 + 2b: predecir residual con el modelo Ridge y aplicar
ridge_full = Ridge(alpha=10.0)
ridge_full.fit(X_sc, y)
predicted_residuals = ridge_full.predict(X_sc)
corrected_error_sec = y - predicted_residuals  # residual después de corrección
mae_capa1_2b = np.abs(corrected_error_sec).mean() / 60

# Capa 2a sola: MAE del prior demográfico (referencia)
# Predecir usando el tiempo oficial medio del grupo age × gender en Boston
boston_group_median = df_bv.groupby(['age_group', gender_col], observed=True)['Official Time_sec'].median()
df_bv['demo_prior_sec'] = df_bv.apply(
    lambda r: boston_group_median.get((r['age_group'], r[gender_col]), np.nan), axis=1
)
mae_capa2a = df_bv.dropna(subset=['demo_prior_sec'])['riegel_residual_sec'].apply(
    lambda x: abs(x)
).mean() / 60  # recompute properly
valid_2a = df_bv.dropna(subset=['demo_prior_sec'])
mae_capa2a = mean_absolute_error(
    valid_2a['Official Time_sec'], valid_2a['demo_prior_sec']
) / 60

print('━' * 60)
print('COMPARACIÓN DE MAE POR CAPA — Boston 2015-2018 (Half → Full)')
print('━' * 60)
print(f'  Referencia global (media global)        : ~{df_bv["Official Time_sec"].std()/60:.0f} min  (sin modelo)')
print(f'  Capa 2a: prior demográfico (age×gender)  : {mae_capa2a:.1f} min  (sin PR)')
print(f'  Capa 1:  Riegel calibrado                : {mae_capa1:.1f} min  (con PR)')
print(f'  Capa 1+2b: Riegel + corrección demográf. : {mae_capa1_2b:.1f} min  (con PR + age/gender)')
print('━' * 60)
mejora = mae_capa1 - mae_capa1_2b
print(f'\n  Mejora de Capa 2b sobre Capa 1 sola: {mejora:+.1f} min')
print(f'  Mejora de Capa 1 sobre Capa 2a:      {mae_capa2a - mae_capa1:+.1f} min')
print('\n  → El PR (Riegel) domina la predicción cuando está disponible.')
print('  → La corrección demográfica aporta ajuste fino (~1-2 min).')
print('  → Sin PR, el prior demográfico reduce el error vs media global.')

In [ ]:
# ─── Figura 04_02: sesgo de Riegel por edad × género ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Capa 2b — Sesgo de Riegel por edad y género (Boston 2015-2018)\nResiduos positivos = Riegel optimista (corredor llegó más tarde de lo predicho)', fontsize=11)

bias_by_group = df_bv.groupby(['age_group', gender_col], observed=True)['riegel_residual_min'].agg(['mean', 'count'])
mae_by_group  = df_bv.groupby(['age_group', gender_col], observed=True)['riegel_residual_min'].agg(lambda x: x.abs().mean())

for ax, gender, color, label in zip(axes, ['M', 'F'], ['#2196F3', '#E91E63'], ['Hombres', 'Mujeres']):
    gdata = bias_by_group.xs(gender, level=1) if gender in bias_by_group.index.get_level_values(1) else pd.DataFrame()
    gmae  = mae_by_group.xs(gender, level=1)  if gender in mae_by_group.index.get_level_values(1) else pd.Series(dtype=float)

    x = range(len(gdata))
    bars = ax.bar(x, gdata['mean'].values, color=color, alpha=0.7, label='Sesgo (media)')
    ax.plot(x, gmae.values, 'ko--', ms=5, lw=1.5, label='MAE')
    ax.axhline(0, color='black', lw=1)
    ax.set_xticks(list(x))
    ax.set_xticklabels(gdata.index.tolist(), rotation=30, ha='right')
    ax.set_ylabel('Minutos')
    ax.set_xlabel('Grupo de edad')
    ax.set_title(f'{label}: sesgo medio y MAE de Riegel por grupo')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Anotar n por grupo
    for xi, (idx, row) in zip(x, gdata.iterrows()):
        ax.text(xi, row['mean'] + 0.3, f'n={int(row["count"]//100)*100}+', 
                ha='center', va='bottom', fontsize=7, rotation=90, alpha=0.6)

plt.tight_layout()
fig_path = FIGS_DIR / '04_fig_02_sesgo_riegel_demografico.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path.name}')

---
## 5. Cobertura para distancias cortas (5K, 10K, 21K)

Los splits de Boston 2015-2018 incluyen tiempos acumulados a 5K, 10K y la media maratón. Esto permite **evaluar Riegel** desde esas distancias hacia el tiempo final, pero con una advertencia importante.

In [ ]:
# ─── Riegel desde cada split hacia el tiempo oficial ─────────────────────────
SPLITS_TO_TEST = [
    ('5K',  5.0,      '5K_sec'),
    ('10K', 10.0,     '10K_sec'),
    ('20K', 20.0,     '20K_sec'),
    ('Half', D_HALF,  'Half_sec'),
]

print('MAE de Riegel calibrado desde cada split hacia el tiempo oficial:')
print('━' * 70)
print(f'{'Desde':8s}  {'n':>7s}  {'MAE (min)':>10s}  {'Sesgo (min)':>12s}  {'Nota'}')
print('─' * 70)

for split_name, split_km, col_sec in SPLITS_TO_TEST:
    if col_sec not in df_bv.columns:
        continue
    sub = df_bv[
        df_bv[col_sec].notna() &
        df_bv[col_sec].between(split_km * 180, split_km * 600)  # rango plausible seg/km
    ].copy()
    if len(sub) < 100:
        continue
    riegel_from_split = sub[col_sec].apply(
        lambda t: riegel_calibrated(t, split_km, D_FULL)[0]
    )
    residual_min = (sub['Official Time_sec'] - riegel_from_split) / 60
    mae_  = residual_min.abs().mean()
    bias_ = residual_min.mean()
    nota  = '⚠️ split en-carrera ≠ PR independiente' if split_name in ['5K', '10K'] else '✅ validación principal'
    print(f'{split_name:8s}  {len(sub):7,}  {mae_:10.1f}  {bias_:+12.1f}  {nota}')

print('─' * 70)
print()
print('INTERPRETACIÓN:')
print('• Los splits a 5K y 10K dentro del maratón son tiempos CONSERVADORES.')
print('  Un corredor que llega a 5K en 28min puede tener un PR de 5K de 22min.')
print('  Riegel predice mejor desde puntos más tardíos (15K, 20K, Half)')
print('  porque ahí el corredor ya mostró su ritmo real de carrera.')
print()
print('• Para el sistema de predicción de la APP:')
print('  - PR de 5K independiente → Riegel predice 10K/21K/42K')
print('  - PR de 21K → predice 42K con el menor MAE (más información sobre resistencia)')
print('  - Sin PR → fallback a prior demográfico')

---
## 6. Viabilidad de la Capa 3 — Corrección por carga de entrenamiento

La Capa 3 necesita un dataset con: **(historial de entrenamiento semanal + tiempo de carrera)** del mismo atleta. Evaluamos qué tenemos.

In [ ]:
# ─── 16620238: confirmar que NO hay race times ────────────────────────────────
print('=== 16620238 — Schema completo ===')
print(df_16.dtypes.to_string())

print(f'\nColumna "major" (primeras 15 categorías):')
if 'major' in df_16.columns:
    print(df_16['major'].value_counts().head(15).to_string())
    print('\n→ "major" es el nombre de la carrera principal del atleta, NO el tiempo.')
    print('  No permite linkear con resultados de carrera sin un ID común.')

print(f'\nMuestra de filas:')
print(df_16.head(4).to_string())

print('\n[CONCLUSIÓN] 16620238 tiene:')
print('  ✅ datetime, athlete_id, distance_semanal, duration, gender, age_group, country, major')
print('  ❌ NO tiene: tiempo de carrera, PR, pace objetivo, resultado en competencia')
print('  → Útil para CTL/ATL/ACWR features. No para entrenamiento supervisado directo.')

In [ ]:
# ─── Run Club: evaluar si es usable para Capa 3 ──────────────────────────────
run_club_train = DATA_DIR / 'Run Club Marathon Performance Dataset' / 'train.csv'

if run_club_train.exists():
    df_rc = pd.read_csv(run_club_train)
    print(f'Run Club train.csv: {df_rc.shape}')
    print(f'Columnas: {df_rc.columns.tolist()}')

    # Identificar target
    target_candidates = [c for c in df_rc.columns
                         if any(kw in c.lower() for kw in ['finish','time','marathon','target'])]
    print(f'\nCandidatos a target: {target_candidates}')

    for tc in target_candidates[:2]:
        print(f'\n{tc}:')
        print(df_rc[tc].describe().to_string())

    # Test de sinteticidad
    print('\n--- Test de sinteticidad ---')
    numeric_cols = df_rc.select_dtypes('number').columns[:8]
    for col in numeric_cols:
        uniq = df_rc[col].nunique()
        rng  = df_rc[col].max() - df_rc[col].min()
        std  = df_rc[col].std()
        # Señal de sinteticidad: muy pocos valores únicos para variable continua
        flag = '⚠️ POCAS CATEGORÍAS' if uniq < 20 and 'int' not in str(df_rc[col].dtype) else ''
        print(f'  {col:35s}: {uniq:5d} únicos  std={std:.2f}  {flag}')
else:
    print('Run Club train.csv: no encontrado en el path esperado.')
    print(f'Buscando en: {run_club_train}')
    # Buscar en subdirectorios
    for f in (DATA_DIR / 'Run Club Marathon Performance Dataset').glob('*.csv'):
        print(f'  Encontrado: {f.name}  ({f.stat().st_size//1024} KB)')

In [ ]:
# ─── Evaluación de opciones para Capa 3 ──────────────────────────────────────
capa3_options = [
    {
        'Dataset':       'Run Club Marathon Performance',
        'Filas':         '40K (train+test)',
        'Features':      'weekly_km, long_run, pace, vo2max, resting_hr, bmi',
        'Target':        'finish_time_marathon',
        'Estado':        '⚠️ Posiblemente sintético',
        'Acción':        'Evaluar distribuciones; usar con disclaimer en tesis',
    },
    {
        'Dataset':       '16620238 + Boston results (join por major)',
        'Filas':         '~103K (si join es posible)',
        'Features':      'CTL/ATL/ACWR calculados de historial semanal',
        'Target':        'finish_time_boston (si se logra join por nombre)',
        'Estado':        '🔍 Requiere investigar join por nombre/año',
        'Acción':        'Explorar en NB05: ¿hay nombres en ambos datasets?',
    },
    {
        'Dataset':       'Atleta propio (Strava + PRs reales)',
        'Filas':         '1 atleta, datos longitudinales',
        'Features':      'CTL/ATL/ACWR del pipeline actual',
        'Target':        'PR conocido (5100s 21K)',
        'Estado':        '✅ Datos reales, n=1',
        'Acción':        'Caso de estudio en tesis: ejemplo de aplicación del sistema',
    },
    {
        'Dataset':       'Injury Prediction (74 atletas)',
        'Filas':         '42K filas (ventanas temporales)',
        'Features':      'km/zona, RPE, recuperación, fuerza',
        'Target':        'injury (binario)',
        'Estado':        '✅ Real, objetivo secundario',
        'Acción':        'NB04b o NB06: validar acwr_zone() como clasificador',
    },
]

df_capa3 = pd.DataFrame(capa3_options)
print('Opciones para la Capa 3 (corrección por carga de entrenamiento):')
print(df_capa3.to_string(index=False))

---
## 7. Síntesis: qué queda resuelto y qué queda pendiente

In [ ]:
# ─── Tabla final de estado del sistema ───────────────────────────────────────
estado_final = pd.DataFrame([
    # Capa 1
    {'Componente': 'Capa 1: Riegel original (exp=1.06)',
     'Estado': '✅ Completo', 'MAE': '~9.2 min', 'Notebook': 'NB01',
     'Fuente': 'Derivado / NB03'},
    {'Componente': 'Capa 1: Riegel calibrado por segmento',
     'Estado': '✅ Completo', 'MAE': '2.5–12 min', 'Notebook': 'NB03',
     'Fuente': 'Boston 2015-2018'},
    # Capa 2
    {'Componente': 'Capa 2a: Prior demográfico 42K',
     'Estado': '✅ Completo', 'MAE': '~35-45 min', 'Notebook': 'NB04',
     'Fuente': 'Results.csv (429K)'},
    {'Componente': 'Capa 2a: Prior demográfico 5K/10K/21K',
     'Estado': '❌ Sin datos', 'MAE': 'N/A', 'Notebook': '—',
     'Fuente': 'No existe dataset de 5K/10K/21K standalone a escala'},
    {'Componente': 'Capa 2b: Corrección residuos Riegel (Half→Full)',
     'Estado': '✅ Completo', 'MAE': 'mejora ~1-2 min', 'Notebook': 'NB04',
     'Fuente': 'Boston 2015-2018 splits'},
    # Capa 3
    {'Componente': 'Capa 3: Corrección por carga (CTL/ATL/ACWR)',
     'Estado': '⚠️ Pendiente', 'MAE': 'TBD', 'Notebook': 'NB05',
     'Fuente': 'Run Club (evaluar) o atleta propio'},
    # Capa 4
    {'Componente': 'Capa 4: Ajuste por bienestar (check-in)',
     'Estado': '⚠️ Pendiente', 'MAE': 'TBD', 'Notebook': 'NB06+',
     'Fuente': 'Datos del atleta en la app (longitudinal)'},
    # Secundario
    {'Componente': 'Objetivo secundario: ACWR → riesgo lesión',
     'Estado': '⚠️ Pendiente', 'MAE': 'AUC-ROC', 'Notebook': 'NB05',
     'Fuente': 'Injury Prediction dataset (74 atletas)'},
])

print('ESTADO ACTUAL DEL SISTEMA DE PREDICCIÓN MULTI-DISTANCIA')
print('=' * 90)
print(estado_final.to_string(index=False))

---
## 8. Conclusiones

### Qué queda demostrado en este notebook

1. **Cobertura real de distancias:** `Results.csv` cubre **solo maratón (42K)**. No existen en los datasets actuales colecciones de 5K, 10K o 21K standalone a escala comparable. Esto es una limitación honesta del proyecto que debe declararse en la tesis.

2. **Prior demográfico (Capa 2a):** construido con 429K corredores. MAE ~35-45 min. Útil como fallback cuando no hay PR, pero mucho peor que Riegel cuando hay PR disponible (~9 min). Muestra claramente que **el PR es la señal más valiosa del sistema**.

3. **Corrección de residuos (Capa 2b):** con 103K corredores de Boston, el sesgo de Riegel por edad×género está bien caracterizado. La corrección Ridge mejora ~1-2 min sobre Riegel solo. Contribución modesta pero estadísticamente robusta con n grande.

4. **Splits de Boston como proxy:** los tiempos a 5K, 10K y Half en ruta del maratón NO son equivalentes a PRs independientes. Riegel desde 5K de maratón tiene MAE más alto que desde Half. Esto refuerza el diseño del sistema: **preferir siempre el split más tardío disponible**.

5. **Capa 3 — problema de datos:** `16620238` no tiene race times. El dataset Run Club debe evaluarse por sinteticidad antes de usar. La opción más honesta para la tesis es un caso de estudio con el atleta propio.

### Siguiente notebook natural

**Opción A (recomendada):** `05_capa3_carga_y_riesgo_lesion.ipynb`
- Evaluar Run Club para Capa 3 (¿es usable a pesar de la sospecha de sinteticidad?)
- Evaluar Injury Prediction dataset para el objetivo secundario de riesgo de lesión
- Definir si la Capa 3 se puede construir con datos actuales o necesita datos propios

**Opción B (paralela):** Integrar Capa 1 + Capa 2 en el endpoint `predict_from_profile_calibrated()` del pipeline de la app — esto cierra el loop entre la tesis y el producto.